# 01. Roles Processing

This notebook prepares the node-level metadata used throughout the project.

It reads the raw role table, extracts family information from the `Role` field, derives an `arrested` disruption flag from `Request`, validates the transformation logic, and exports the processed file to `data/processed/roles.csv`.

## Inputs and Outputs

- Input: `../data/raw/Montagna_Roles.csv`
- Output: `../data/processed/roles.csv`

Derived columns:

- `family_role`: role or rank extracted from the raw `Role` text
- `family`: family name extracted from quoted text when present
- `arrested`: `1` for `arrested`, ` arrested`, `house arrest`, or `in jail`; otherwise `0`

In [4]:
from pathlib import Path

import pandas as pd

In [5]:
INPUT_PATH = Path('../../data/raw/Montagna_Roles.csv')
OUTPUT_PATH = Path('../../data/processed/roles.csv')

roles = pd.read_csv(INPUT_PATH)
roles.head()

,Node,Role,Relationship,Request
0,N0,cooperating witness,NaN,NaN
1,N1,NaN,conversation with 0,NaN
2,N2,boss family ''Barcellona Pozzo di Gotto'',NaN,NaN
3,N3,boss family ''Caltagirone'',NaN,NaN
4,N4,enterpreneur,NaN,NaN


## Family Parsing

The project uses a simple rule-based parser aligned with the project notes:

- if the word `family` appears, everything before it becomes `family_role`
- if quotes appear, the content inside the quotes becomes `family`
- if neither appears, keep the original role text and mark the family as `Unknown`
- if the source role is missing, keep both derived fields missing

In [6]:
role_text = roles['Role'].astype('string').str.strip()

roles['family_role'] = role_text.str.extract(r'^(.*?)\s*family\b', expand=False)
roles['family_role'] = roles['family_role'].fillna(role_text)
roles['family_role'] = roles['family_role'].str.strip().replace('', pd.NA)

family_from_quotes = role_text.str.extract(r'"([^"]+)"|\'\'([^\']+)\'\'', expand=True)
roles['family'] = family_from_quotes.bfill(axis=1).iloc[:, 0]
roles['family'] = roles['family'].fillna('Unknown')

missing_role_mask = role_text.isna() | role_text.eq('')
roles.loc[missing_role_mask, ['family_role', 'family']] = pd.NA

roles[['Node', 'Role', 'family_role', 'family']].head(20)

,Node,Role,family_role,family
0,N0,cooperating witness,cooperating witness,Unknown
1,N1,NaN,<NA>,<NA>
2,N2,boss family ''Barcellona Pozzo di Gotto'',boss,Barcellona Pozzo di Gotto
3,N3,boss family ''Caltagirone'',boss,Caltagirone
4,N4,enterpreneur,enterpreneur,Unknown
5,N5,NaN,<NA>,<NA>
6,N6,NaN,<NA>,<NA>
7,N7,NaN,<NA>,<NA>
8,N8,NaN,<NA>,<NA>
9,N9,NaN,<NA>,<NA>


## Validation of Role and Family Extraction

These checks make the notebook safer to rerun after future edits.

In [7]:
test_cases = {
    "boss family ''Caltagirone''": ('boss', 'Caltagirone'),
    'boss family "Caltagirone"': ('boss', 'Caltagirone'),
    "member of family ''San Mauro Castelverde''": ('member of', 'San Mauro Castelverde'),
    "Co-founder of family ''Batanesi''": ('Co-founder of', 'Batanesi'),
    'cooperating witness': ('cooperating witness', 'Unknown'),
    'executive family': ('executive', 'Unknown'),
}

for raw_role, expected in test_cases.items():
    sample = pd.Series([raw_role], dtype='string').str.strip()
    parsed_role = sample.str.extract(r'^(.*?)\s*family\b', expand=False).fillna(sample).iloc[0]
    family_match = sample.str.extract(r'"([^"]+)"|\'\'([^\']+)\'\'', expand=True)
    parsed_family = family_match.bfill(axis=1).iloc[0, 0]
    parsed_family = parsed_family if pd.notna(parsed_family) else 'Unknown'
    assert (parsed_role, parsed_family) == expected

roles.loc[[0, 2, 3, 11, 12, 17, 22, 25], ['Node', 'Role', 'family_role', 'family']]

,Node,Role,family_role,family
0,N0,cooperating witness,cooperating witness,Unknown
2,N2,boss family ''Barcellona Pozzo di Gotto'',boss,Barcellona Pozzo di Gotto
3,N3,boss family ''Caltagirone'',boss,Caltagirone
11,N11,boss Cosa Nostra in Messina,boss Cosa Nostra in Messina,Unknown
12,N12,member family ''Mistretta'',member,Mistretta
17,N18,executive family ''Mistretta'',executive,Mistretta
22,N23,executive family,executive,Unknown
25,N26,member of family ''San Mauro Castelverde'',member of,San Mauro Castelverde


In [8]:
roles[['family_role', 'family']].value_counts(dropna=False).head(20)

family_role                  family                   
<NA>                         <NA>                         59
enterpreneur                 Unknown                      30
member                       Batanesi                     12
                             Mistretta                     6
executive                    Mistretta                     3
                             Batanesi                      3
member                       Mazzaroti                     2
boss                         Tortorici                     2
lawyer                       Unknown                       2
road haulier                 Unknown                       2
cooperating witness          Unknown                       1
boss                         Barcellona Pozzo di Gotto     1
                             Caltagirone                   1
boss Cosa Nostra in Messina  Unknown                       1
boss                         Mazzaroti                     1
                             B

## Arrest Flag for Disruption Analysis

This binary field identifies actors already removed or constrained by law enforcement, which is the basis for the post-disruption network in the main analysis notebook.

In [9]:
request_text = roles['Request'].astype('string').str.strip().str.lower()
arrest_values = {'house arrest', 'arrested', 'in jail'}
roles['arrested'] = request_text.isin(arrest_values).astype(int)

roles[['Node', 'Request', 'arrested']].head(20)

,Node,Request,arrested
0,N0,NaN,0
1,N1,NaN,0
2,N2,NaN,0
3,N3,NaN,0
4,N4,NaN,0
5,N5,NaN,0
6,N6,NaN,0
7,N7,NaN,0
8,N8,NaN,0
9,N9,NaN,0


In [10]:
assert roles.loc[roles['Request'].eq('arrested'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq(' arrested'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq('in jail'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq('house arrest'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq('arrest request denied'), 'arrested'].eq(0).all()

roles['arrested'].value_counts(dropna=False)

arrested
0    102
1     41
Name: count, dtype: int64

## Export Processed Metadata

The processed output will be used by the full network notebook and any downstream reports.

In [11]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
roles.to_csv(OUTPUT_PATH, index=False)

pd.DataFrame(
    {
        'rows': [len(roles)],
        'unique_nodes': [roles['Node'].nunique()],
        'arrested_nodes': [int(roles['arrested'].sum())],
        'known_families': [int(roles['family'].fillna('Unknown').ne('Unknown').sum())],
        'output_path': [str(OUTPUT_PATH)],
    }
)

,rows,unique_nodes,arrested_nodes,known_families,output_path
0,143,143,41,36,../../data/processed/roles.csv
